# Build a Basic Chatbot with LangGraph (Graph API)

Training reference — **LangGraph crash course**: nodes, edges, state.

**[reference/langgraph-crash-course.png](reference/langgraph-crash-course.png)**

![LangGraph crash course — nodes, edges, state, YT URL to transcript, title, content, finalizer](reference/langgraph-crash-course.png)

| Concept | In this chatbot lab |
|---------|---------------------|
| **Nodes** | `llmchatbot` — calls the LLM |
| **Edges** | `START → llmchatbot → END` |
| **State** | `messages` list (chat history) |

**Prerequisite:** `GROQ_API_KEY` in repo-root `.env` · `uv sync`

**Sections:** 1 LangGraph basics | 2 imports | 3 state | 4 `StateGraph` | 5 LLM | 6 node | 7 compile | 8 visualize | 9 invoke | 10 stream | 11 summary

## 1. What is LangGraph?

**LangGraph** builds **stateful workflows** as a **graph**:

| Term | Meaning |
|------|---------|
| **Node** | A Python function — one step (call LLM, fetch data, etc.) |
| **Edge** | Connection between nodes — defines execution order |
| **State** | Shared dict every node reads and updates |

This lab uses the **Graph API** (`StateGraph`) — you wire nodes and edges yourself (unlike `create_agent`, which builds the loop for you).

```
START → llmchatbot → END
```

## 2. Imports — what each module does

| Import | Package | Purpose |
|--------|---------|---------|
| `Annotated` | `typing` | Metadata on type hints — tells LangGraph *how* to merge state updates |
| `TypedDict` | `typing_extensions` | Defines **`State`** keys and types |
| `StateGraph` | `langgraph.graph` | Graph **builder** — add nodes/edges, then `.compile()` |
| `START` / `END` | `langgraph.graph` | Entry and exit pseudo-nodes |
| `add_messages` | `langgraph.graph.message` | **Reducer** — append new chat messages instead of replacing the list |
| `ChatGroq` | `langchain_groq` | Groq-hosted LLM (fast, free tier for learning) |
| `HumanMessage` | `langchain_core.messages` | Wraps user text in the message format the LLM expects |

In [ ]:
# --- Standard library ---
from typing import Annotated

# --- State schema ---
from typing_extensions import TypedDict

# --- LangGraph Graph API ---
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# --- LLM + messages (used in later cells) ---
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

## 3. Define `State` — shared memory

Each node receives `state` and returns a **partial update**. LangGraph merges it into the full state.

| Piece | Role |
|-------|------|
| `class State(TypedDict)` | Allowed keys on state |
| `messages: Annotated[list, add_messages]` | Chat history — **append** on each turn |

Without `add_messages`, returning `{"messages": [new]}` would **wipe** prior turns.

In [ ]:
class State(TypedDict):
    """Shared state passed between all graph nodes."""
    messages: Annotated[list, add_messages]

## 4. Create `StateGraph` — the builder

`StateGraph(State)` creates an **empty** graph template bound to your state schema. You add nodes and edges next, then `.compile()`.

| Method | What it does |
|--------|----------------|
| `add_node(name, fn)` | Register a Python function as a node |
| `add_edge(a, b)` | Run `b` after `a` finishes |
| `compile()` | Produce a runnable `CompiledStateGraph` |

In [ ]:
graph_builder = StateGraph(State)
print("StateGraph builder ready — nodes not added yet.")

## 5. Load API key and initialize the LLM

We use **Groq** with `qwen/qwen3-32b` (same model as other tutorials in this repo).

The LLM is used **inside** the chatbot node — not by LangGraph directly.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    print("GROQ_API_KEY: missing (add to repo-root .env)")

llm = ChatGroq(model="qwen/qwen3-32b")
print("LLM:", llm.model_name)

## 6. Define the chatbot **node**

A **node** is a plain function: `state in → partial state update out`.

```python
def chatbot(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}
```

| Step | What happens |
|------|----------------|
| Read | `state["messages"]` — full chat history so far |
| Call | `llm.invoke(...)` — Groq returns an `AIMessage` |
| Return | `{"messages": [response]}` — `add_messages` **appends** the AI reply |

This is the only node in our minimal graph.

In [ ]:
def chatbot(state: State) -> dict:
    """Call the LLM with current messages; append AI response to state."""
    return {"messages": [llm.invoke(state["messages"])]}

## 7. Wire the graph — nodes, edges, compile

| Step | Code | Result |
|------|------|--------|
| 1 | `add_node("llmchatbot", chatbot)` | Register the node |
| 2 | `add_edge(START, "llmchatbot")` | Entry → chatbot |
| 3 | `add_edge("llmchatbot", END)` | chatbot → exit |
| 4 | `compile()` | Runnable graph |

Matches the training diagram: one linear path from input to output.

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("llmchatbot", chatbot)
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

graph = graph_builder.compile()
print("Graph compiled:", type(graph).__name__)

## 8. Visualize the graph (optional)

`graph.get_graph().draw_mermaid_png()` renders a PNG in Jupyter. Requires network or a local Mermaid renderer — wrapped in `try/except` so the notebook still runs offline.

**Tip:** Use the static training image at the top if this cell shows nothing.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Graph viz skipped (offline or no Mermaid renderer):", type(exc).__name__)

## 9. Run the graph — `.invoke()`

Pass initial state. Use `HumanMessage` for user text (recommended) or a plain string (LangGraph coerces it).

```python
graph.invoke({"messages": [HumanMessage(content="Hi")]})
```

Returns final state including full `messages` list (user + AI).

In [ ]:
response = graph.invoke({
    "messages": [HumanMessage(content="Hi")],
})

print("Last reply:")
print(response["messages"][-1].content)

In [ ]:
# Full message list: HumanMessage + AIMessage
response["messages"]

## 10. Stream tokens — `.stream()`

`.stream()` yields **events after each node** — useful for progress UI or multi-node graphs.

For this single-node chatbot, each event contains the updated `messages` after `llmchatbot` runs.

In [ ]:
for event in graph.stream({"messages": [HumanMessage(content="Hi, how are you?")]}):
    for value in event.values():
        print(value["messages"][-1].content)

## 11. Notebook summary

### What you built

```
START → llmchatbot (LLM) → END
         State.messages grows each turn
```

### API cheat sheet

| Task | Code |
|------|------|
| Define state | `class State(TypedDict): messages: Annotated[list, add_messages]` |
| Build graph | `StateGraph(State)` → `add_node` → `add_edge` → `compile()` |
| Run once | `graph.invoke({"messages": [HumanMessage(...)]})` |
| Stream steps | `graph.stream({...})` |
| Node pattern | `def node(state): return {"messages": [llm.invoke(state["messages"])]}` |

### LangGraph vs LangChain agent

| | **This notebook (LangGraph)** | **`create_agent` (notebook 1)** |
|---|------------------------------|--------------------------------|
| Control | You define every node and edge | Framework runs model ⇄ tools loop |
| Best for | Custom workflows, branching, persistence | Quick agents with tools |

### Next steps

- **Memory:** add `checkpointer=InMemorySaver()` + `thread_id` for multi-turn sessions (see [6_middleware.ipynb](../../updated_langchain_tutorial/6_middleware.ipynb))
- **Tools:** add a tool node and conditional edges (ReAct-style loop)
- **Messages:** [4_messages.ipynb](../../updated_langchain_tutorial/4_messages.ipynb)

**Training diagram:** [reference/langgraph-crash-course.png](reference/langgraph-crash-course.png)